**<h1>Electric Line Extension - Analysis 3**
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 09/23/2025 | Start Development: 09/08/2025</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for Q1 2025
* Goals: 1. Clean excel files for public downloads. See 'Clean 3' below for full details and results


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Load dataframes
pge_q1_2025 = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/PGE Q1 2025 Data.xlsx", header=3)
sdge_q1_2025 = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SDG&E Q1 2025 Data.xlsx", header=3)
sce_q1_2025 = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw/SCE Q1 2025 Data.xlsx", header=6)

In [3]:
# Check the dataframes
print(sdge_q1_2025.shape)
print(sce_q1_2025.shape)
print(pge_q1_2025.shape)    

print (sdge_q1_2025.columns)
print (sce_q1_2025.columns)
print (pge_q1_2025.columns)


(219, 18)
(312, 18)
(112, 18)
Index(['New Construction / Upgrade', 'Mixed Fuel / All Electric', 'Months',
       'Customer Class', 'Baseline / Climate Zone', 'Multi Dwelling',
       'Total Discounts (Non-Exempted Projects)',
       'Total Allowance (Non-Exempted Projects)',
       'Total Refund Payments Provided to Builders  (Non-Exempted Projects)',
       'Total Discounts (Exempted Projects)',
       'Total Allowance (Exempted Projects)',
       'Total Refund Payments Provided to Builders  (Exempted Projects)',
       'Total Estimated Non-Refundable', 'Total Estimated Refundable',
       'Average Number of days between full payment and Project Energization',
       'Total Electric Line Extension Energized',
       'Total Electric Line Extension Applications (IOU Installed)',
       'Total Electric Line Extension Applications (Applicant Installed)'],
      dtype='object')
Index(['New Construction /\nUpgrade', 'Mixed Fuel / All Electric', 'Month',
       'Customer Class', 'Baseline / 

**CLEAN 3**

* For PGE:
    *   Map the fuel type and construction type to the correct naming convention as other IOUs
* For All IOUs: Create dataframes from main sheet:
    *   Rename the 'Months' column to read 'Month' for SDGE/PGE and remove new lines and extra spaces where necessary
    *   Use the fuel type and construction type to recreate disctionaries in the same format as 2023 & 2024 data
    *   Extract the individual fuel/structure types into dataframes: All Electric NC, Mixed Fuel NC, All Electric Upgrades, Mixed Fuel Upgrades 
    *   Create melted dataframes using the melt function from previous iterations, which groups by customer class and melts on class, month, climate zone, and dwelling unit type. 
    *   Access specific  fuel/structure types and standardize the naming of months.
    *   Excise unnecessary rows per dataframe and create a new column to capture the fuel/structure type
    *   Concatonate the dataframes per IOU; these are exported as excel spreadsheets and moved to the 'processed' data folder. The melted exports are housed in the 'interim' data folder

In [4]:
# Standardize Month column name
sdge_q1_2025 = sdge_q1_2025.rename(columns={'Months': 'Month'})
pge_q1_2025 = pge_q1_2025.rename(columns={'Months': 'Month'})

In [5]:
# Clean up column names: remove newlines and extra spaces
def clean_columns(df):
    df.columns = [col.replace('\n', ' ').replace('\r', ' ').strip() for col in df.columns]
    return df

pge_q1_2025 = clean_columns(pge_q1_2025)
sdge_q1_2025 = clean_columns(sdge_q1_2025)
sce_q1_2025 = clean_columns(sce_q1_2025)

In [6]:
fuel_types = ['All Electric', 'Mixed Fuel']
structure_types = ['New Construction', 'Upgrade']

In [7]:
# For each utility, create a dictionary with keys like 'All Electric New Construction'
def split_by_fuel_and_structure(df):
    split_dfs = {}
    for fuel in fuel_types:
        for struct in structure_types:
            key = f"{fuel} {struct}"
            split_dfs[key] = df[
                (df['Mixed Fuel / All Electric'] == fuel) &
                (df['New Construction / Upgrade'] == struct)
            ].copy()
    return split_dfs

sdge_split = split_by_fuel_and_structure(sdge_q1_2025)
sce_split = split_by_fuel_and_structure(sce_q1_2025)

In [8]:
# Map PGE's fuel types to match SDGE/SCE
pge_q1_2025['Mixed Fuel / All Electric'] = pge_q1_2025['Mixed Fuel / All Electric'].replace({'Elec': 'All Electric', 'Dual': 'Mixed Fuel'})
pge_q1_2025['New Construction / Upgrade'] = pge_q1_2025['New Construction / Upgrade'].replace({'Existing Construction': 'Upgrade'})
# Now you can use the same split function as before
pge_split = split_by_fuel_and_structure(pge_q1_2025)

In [9]:
def melt_by_customer_class(df, customer_class_col='Customer Class', month_col='Month', climate_zone='Baseline / Climate Zone', dwtype='Multi Dwelling'):
    id_vars = [month_col, customer_class_col, climate_zone, dwtype]
    if df.empty or any(col not in df.columns for col in id_vars):
        return pd.DataFrame()
    return df.melt(id_vars=id_vars, var_name='Type', value_name='Value')

In [10]:
pge_melted = {k: melt_by_customer_class(df) for k, df in pge_split.items()}
sdge_melted = {k: melt_by_customer_class(df) for k, df in sdge_split.items()}
sce_melted = {k: melt_by_customer_class(df) for k, df in sce_split.items()}

In [11]:
# Access specific melted DataFrames
pge_MFNC = pge_melted['Mixed Fuel New Construction']
pge_AENC = pge_melted['All Electric New Construction']
pge_MFUP = pge_melted['Mixed Fuel Upgrade']
pge_AEUP = pge_melted['All Electric Upgrade']
sdge_MFNC = sdge_melted['Mixed Fuel New Construction']
sdge_AENC = sdge_melted['All Electric New Construction']
sdge_MFUP = sdge_melted['Mixed Fuel Upgrade']
sdge_AEUP = sdge_melted['All Electric Upgrade']
sce_MFNC = sce_melted['Mixed Fuel New Construction']
sce_AENC = sce_melted['All Electric New Construction']
sce_MFUP = sce_melted['Mixed Fuel Upgrade']
sce_AEUP = sce_melted['All Electric Upgrade']

In [12]:
def format_month_column(df):
    # If values look like dates (e.g., '2025-01-01'), convert to 'Jan 2025'
    if pd.to_datetime(df['Month'], errors='coerce').notna().all():
        df['Month'] = pd.to_datetime(df['Month']).dt.strftime('%b %Y')
    else:
        # If values are just month names (e.g., 'January'), add year and abbreviate
        month_map = {
            'January': 'Jan', 'February': 'Feb', 'March': 'Mar', 'April': 'Apr',
            'May': 'May', 'June': 'Jun', 'July': 'Jul', 'August': 'Aug',
            'September': 'Sep', 'October': 'Oct', 'November': 'Nov', 'December': 'Dec'
        }
        df['Month'] = df['Month'].apply(lambda x: f"{month_map.get(str(x), str(x)[:3])} 2025")
    return df

# Apply to each melted dataframe
for df in [pge_MFNC, pge_AENC, pge_MFUP, pge_AEUP, sdge_MFNC, sdge_AENC, sdge_MFUP, sdge_AEUP, sce_MFNC, sce_AENC, sce_MFUP, sce_AEUP]:
    df = format_month_column(df)

C:\Users\Rford\AppData\Local\Temp\ipykernel_55128\423870880.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  if pd.to_datetime(df['Month'], errors='coerce').notna().all():
C:\Users\Rford\AppData\Local\Temp\ipykernel_55128\423870880.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  if pd.to_datetime(df['Month'], errors='coerce').notna().all():
C:\Users\Rford\AppData\Local\Temp\ipykernel_55128\423870880.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  if pd.to_datetime(df['Month'], errors='coerce').notna().all():
C:\Users\Rford\AppData\Local\Temp\ipykernel_55128\42387

In [13]:
# Remove initial rows and add the combo column 'Fuel_Structure_type' to represnet them for each dataframe 
pge_MFNC = pge_MFNC.iloc[30:].reset_index(drop=True)  # Remove first 31 rows
pge_MFNC['Fuel_Structure_type'] = 'Mixed Fuel New Construction'
pge_AENC = pge_AENC.iloc[54:].reset_index(drop=True)  # Remove first 54 rows
pge_AENC['Fuel_Structure_type'] = 'All Electric New Construction'
pge_MFUP = pge_MFUP.iloc[56:].reset_index(drop=True)  # Remove first 56 rows
pge_MFUP['Fuel_Structure_type'] = 'Mixed Fuel Upgrade'
pge_AEUP = pge_AEUP.iloc[78:].reset_index(drop=True)  # Remove first 78 rows
pge_AEUP['Fuel_Structure_type'] = 'All Electric Upgrade'

sdge_MFNC = sdge_MFNC.iloc[112:].reset_index(drop=True)  # Remove first 112 rows
sdge_MFNC['Fuel_Structure_type'] = 'Mixed Fuel New Construction'
sdge_AENC = sdge_AENC.iloc[138:].reset_index(drop=True)  # Remove first 138 rows
sdge_AENC['Fuel_Structure_type'] = 'All Electric New Construction'
sdge_MFUP = sdge_MFUP.iloc[62:].reset_index(drop=True)  # Remove first 62 rows
sdge_MFUP['Fuel_Structure_type'] = 'Mixed Fuel Upgrade'
sdge_AEUP = sdge_AEUP.iloc[126:].reset_index(drop=True)  # Remove first 126 rows
sdge_AEUP['Fuel_Structure_type'] = 'All Electric Upgrade'

sce_MFNC = sce_MFNC.iloc[164:].reset_index(drop=True)  # Remove first 164 rows
sce_MFNC['Fuel_Structure_type'] = 'Mixed Fuel New Construction'
sce_AENC = sce_AENC.iloc[168:].reset_index(drop=True)  # Remove first 168 rows
sce_AENC['Fuel_Structure_type'] = 'All Electric New Construction'
sce_MFUP = sce_MFUP.iloc[160:].reset_index(drop=True)  # Remove first 160 rows
sce_MFUP['Fuel_Structure_type'] = 'Mixed Fuel Upgrade'
sce_AEUP = sce_AEUP.iloc[130:].reset_index(drop=True)  # Remove first 130 rows
sce_AEUP['Fuel_Structure_type'] = 'All Electric Upgrade'


In [14]:
#Concatenate the dataframes for each utility
pge_2025_composite = pd.concat([pge_MFNC, pge_AENC, pge_MFUP, pge_AEUP], ignore_index=True)
sdge_2025_composite = pd.concat([sdge_MFNC, sdge_AENC, sdge_MFUP, sdge_AEUP], ignore_index=True)
sce_2025_composite = pd.concat([sce_MFNC, sce_AENC, sce_MFUP, sce_AEUP], ignore_index=True)

In [15]:
pge_2025_composite['IoU'] = 'PG&E'
sdge_2025_composite['IoU'] = 'SDG&E'
sce_2025_composite['IoU'] = 'SCE'


In [16]:
print(pge_2025_composite.shape)
print(sdge_2025_composite.shape)
print(sce_2025_composite.shape)

(1308, 8)
(2628, 8)
(3732, 8)


In [18]:
# Save the composite dataframes to Excel
pge_2025_composite.to_excel("PGE 2025 Composite.xlsx", index=False)
sdge_2025_composite.to_excel("SDGE 2025 Composite.xlsx", index=False)
sce_2025_composite.to_excel("SCE 2025 Composite.xlsx", index=False)

# Save each dataframe to Excel based on fuel
#pge_MFNC.to_excel("PGE Mixed Fuel New Construction.xlsx", index=False)
#pge_AENC.to_excel("PGE All Electric New Construction.xlsx", index=False)
#pge_MFUP.to_excel("PGE Mixed Fuel Upgrade.xlsx", index=False)
#pge_AEUP.to_excel("PGE All Electric Upgrade.xlsx", index=False)
#sdge_MFNC.to_excel("SDGE Mixed Fuel New Construction.xlsx", index=False)
#sdge_AENC.to_excel("SDGE All Electric New Construction.xlsx", index=False)
#sdge_MFUP.to_excel("SDGE Mixed Fuel Upgrade.xlsx", index=False)
#sdge_AEUP.to_excel("SDGE All Electric Upgrade.xlsx", index=False)
#sce_MFNC.to_excel("SCE Mixed Fuel New Construction.xlsx", index=False)
#sce_AENC.to_excel("SCE All Electric New Construction.xlsx", index=False)
#sce_MFUP.to_excel("SCE Mixed Fuel Upgrade.xlsx", index=False)
#sce_AEUP.to_excel("SCE All Electric Upgrade.xlsx", index=False)